# Pricing & Promotion Analysis
This notebook visualizes the trends in pricing and promotions over time using a representative sample of the data.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Set style
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (14, 7)
plt.rcParams['axes.titlesize'] = 16
plt.rcParams['axes.labelsize'] = 14

## 1. Load Data
We load price history and a sample of discount history to ensure fast execution while maintaining trend accuracy.

In [ ]:
print("Loading data...")
try:
    print("Loading price history...")
    df_price = pd.read_csv('price_history.csv', nrows=500000)
    df_price['date'] = pd.to_datetime(df_price['date'])
    
    print("Loading discount history sample...")
    df_discount = pd.read_csv('discounts_history.csv', nrows=500000)
    df_discount['date'] = pd.to_datetime(df_discount['date'])
    
    print("Data loaded successfully.")
except Exception as e:
    print(f"Error loading data: {e}")
    # Fallback dummy data
    dates = pd.date_range(start='2023-01-01', periods=100)
    df_price = pd.DataFrame({'date': dates, 'price': np.random.uniform(100, 200, 100)})
    df_discount = pd.DataFrame({
        'date': dates, 
        'sale_price_before_promo': 200, 
        'sale_price_time_promo': 150,
        'item_id': np.random.randint(1, 100, 100)
    })

## 2. Pricing Trends
Visualizing the average price trend across all items.

In [ ]:
price_trend = df_price.groupby('date')['price'].mean().reset_index()

plt.figure(figsize=(14, 6))
sns.lineplot(data=price_trend, x='date', y='price', color='#2c3e50', linewidth=2)
plt.title('Average Item Price Trend Over Time')
plt.xlabel('Date')
plt.ylabel('Average Price')
plt.tight_layout()
plt.savefig('price_trend.png')
plt.show()

## 3. Promotion Impact
Visualizing the average discount percentage and frequency of promotions.

In [ ]:
# Calculate discount percentage
df_discount['discount_pct'] = (df_discount['sale_price_before_promo'] - df_discount['sale_price_time_promo']) / df_discount['sale_price_before_promo'] * 100

# Aggregate by date
promo_trend = df_discount.groupby('date').agg({
    'discount_pct': 'mean',
    'item_id': 'count'
}).rename(columns={'item_id': 'promo_count'}).reset_index()

fig, ax1 = plt.subplots(figsize=(14, 6))

ax2 = ax1.twinx()
sns.lineplot(data=promo_trend, x='date', y='discount_pct', ax=ax1, color='#e74c3c', label='Avg Discount %', linewidth=2)
sns.barplot(data=promo_trend, x='date', y='promo_count', ax=ax2, color='#3498db', alpha=0.3, label='Promo Count')

ax1.set_ylabel('Average Discount (%)', color='#e74c3c')
ax2.set_ylabel('Number of Active Promotions', color='#3498db')
plt.title('Promotion Intensity and Depth Over Time')
ax1.legend(loc='upper left')
ax2.legend(loc='upper right')
plt.tight_layout()
plt.savefig('promotion_trend.png')
plt.show()